##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [4]:
!pip install torch torchvision transformers accelerate scikit-learn Pillow requests --quiet

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import AutoImageProcessor, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

MODEL_ID = "facebook/dinov2-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()

cat_paths = sorted(Path("20 test images/cats").glob("*.jpg"))
dog_paths = sorted(Path("20 test images/dogs").glob("*.jpg"))

image_paths  = cat_paths + dog_paths
ground_truth = [0] * len(cat_paths) + [1] * len(dog_paths)  # 0=cat, 1=dog

print(f"Loaded {len(cat_paths)} cat images and {len(dog_paths)} dog images")

def get_cls_token(path):
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    cls = outputs.last_hidden_state[:, 0]            # CLS token, shape (1, 384)
    return F.normalize(cls, p=2, dim=1).cpu().numpy()

features = np.vstack([get_cls_token(p) for p in image_paths])  # (20, 384)
print(f"Feature matrix: {features.shape}")

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
predicted_clusters = kmeans.fit_predict(features)

acc_direct  = accuracy_score(ground_truth, predicted_clusters)
acc_flipped = accuracy_score(ground_truth, 1 - predicted_clusters)

if acc_flipped > acc_direct:
    predicted_clusters = 1 - predicted_clusters

best_acc = max(acc_direct, acc_flipped)

print(f"\nClustering Accuracy: {best_acc * 100:.1f}%\n")

for path, pred, gt in zip(image_paths, predicted_clusters, ground_truth):
    label = "cat" if gt == 0 else "dog"
    match = "✓" if pred == gt else "✗"
    print(f"  {path.name:<20s} | GT: {label} | Cluster: {pred} | {match}")

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 22452.59it/s]


Loaded 10 cat images and 10 dog images
Feature matrix: (20, 384)

Clustering Accuracy: 95.0%

  Abyssinian_10.jpg    | GT: cat | Cluster: 0 | ✓
  Bengal_199.jpg       | GT: cat | Cluster: 0 | ✓
  Birman_8.jpg         | GT: cat | Cluster: 0 | ✓
  Bombay_192.jpg       | GT: cat | Cluster: 0 | ✓
  British_Shorthair_267.jpg | GT: cat | Cluster: 0 | ✓
  Egyptian_Mau_30.jpg  | GT: cat | Cluster: 0 | ✓
  Maine_Coon_108.jpg   | GT: cat | Cluster: 0 | ✓
  Persian_66.jpg       | GT: cat | Cluster: 0 | ✓
  Russian_Blue_156.jpg | GT: cat | Cluster: 0 | ✓
  Sphynx_155.jpg       | GT: cat | Cluster: 0 | ✓
  american_bulldog_10.jpg | GT: dog | Cluster: 1 | ✓
  basset_hound_98.jpg  | GT: dog | Cluster: 1 | ✓
  beagle_71.jpg        | GT: dog | Cluster: 1 | ✓
  boxer_9.jpg          | GT: dog | Cluster: 1 | ✓
  chihuahua_9.jpg      | GT: dog | Cluster: 0 | ✗
  english_cocker_spaniel_9.jpg | GT: dog | Cluster: 1 | ✓
  english_setter_90.jpg | GT: dog | Cluster: 1 | ✓
  german_shorthaired_42.jpg | GT: dog |

### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [ ]:
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch

MODEL_ID = "facebook/dinov2-small-imagenet1k-1-layer"   
device   = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model     = AutoModelForImageClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

IMAGE_PATH = "data/GR.jpeg"         

img    = Image.open(IMAGE_PATH).convert("RGB")
inputs = processor(images=img, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inputs).logits  
    predicted_class_idx = logits.argmax(-1).item()


top5_scores, top5_ids = torch.topk(logits.softmax(dim=-1), k=5)

print("Top-5 Predictions:")
print("-" * 40)
for rank, (score, class_id) in enumerate(zip(top5_scores[0], top5_ids[0]), start=1):
    label = model.config.id2label[class_id.item()]
    print(f"  {rank}. {label:<30s} {score.item()*100:5.1f}%")

predicted_label = model.config.id2label[predicted_class_idx]
print(f"The model classified this image as: {predicted_label}")


Loading weights: 100%|██████████| 225/225 [00:00<00:00, 21936.23it/s]

Top-5 Predictions:
----------------------------------------
  1. golden retriever                92.8%
  2. curly-coated retriever           1.2%
  3. Leonberg                         0.8%
  4. Sussex spaniel                   0.8%
  5. Irish setter, red setter         0.6%
The model classified this image as: golden retriever
